# LELA60341: Research Methods in Computational and Corpus Linguistics Week 8 · Solutions

## A logistic regression model, built from parts (in class)

In the LELA60331 seminars you have built **logistic regression** step by step, as a long series of notebook cells, and in Week 7 you trained it on simulated data. In this workbook you will build the same classifier again, using what you learned about classes in Week 7, in a way that makes it easy to reuse and to extend. It has three parts:

| Part | What it does |
|---|---|
| a **layer** (the class `Linear`) | turns each example's features into a score (the **forward** pass), works out the gradients for its weights (the **backward** pass), and updates its weights |
| a **loss** (two functions, `bce_loss` and `bce_grad`) | measures how wrong the scores are, and says which way each score should move |
| a **training loop** | puts them together |

By the end of the after-class workbook you will train the model with a loop like this one, in which each line does one job:

```python
for epoch in range(n_epochs):
    scores = model.forward(X)                 # the forward pass
    loss = bce_loss(scores, y)                # how wrong is it?
    model.backward(bce_grad(scores, y))       # the backward pass: the gradients
    model.update(lr)                          # change the weights
```

We build every part in small steps: first by working it out for a tiny example, then by adding it to the class.

This workbook is for the class, and builds the layer. A second workbook, `LELA60341_Wk8_after_class`, is for you to do on your own afterwards: there you write the loss and the training loop, and train the model.

| Part | Topic |
|---|---|
| 1 | From a script to a class |
| 2 | The data |
| 3 | A linear layer, one method at a time |

Read each explanation, run the example cells, then try the 🏋️ exercises. In an exercise, replace each `???` with your own code, then run the ✅ check cell underneath. When you see a 🔮 **Predict** prompt, decide what you think will happen *before* you run the cell. Parts marked ⭐ **Challenge** are optional.

> ✅ **This is the solutions version of the workbook.** Every exercise is completed, and 💡 cells give answers to the 🔮 Predict prompts and discussion questions.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used by every exercise.
def check(label, condition):
    """Print a tick or a cross instead of crashing, so you can keep working."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 1 · From a script to a class

In the LELA60331 seminars, each step of the classifier is a separate cell, and the cells share their results through **global names**. Even for the simulated data in LELA60331 Week 7 there are a lot of them:

| Name | What it holds |
|---|---|
| `x`, `y`, `X` | the features and the labels |
| `weights`, `bias` | the model's parameters |
| `lr`, `n_epochs` | the training settings |
| `q`, `errors`, `loss_history` | the probabilities, the errors and the losses |
| `score`, `sigmoid`, `average_loss`, `mat_vec`, ... | the functions that work on them |

This works, but it is fragile:

* Run the cells in the wrong order, or re-run one by mistake, and the names hold the wrong values. (In LELA60331 Week 7, `q` holds the probabilities from whichever cell you ran last: the untrained model or the trained one.)
* You can't easily have **two** models at once, because they would both want the name `weights`.
* To use the classifier on new data, you have to remember which cells to copy, and in which order.

A class solves these problems. Its **instance variables** keep the model's data together, and its **methods** are the operations on them (Week 7). Each model is one object, with its own weights. For our layer:

| Instance variables (data) | Methods (operations) |
|---|---|
| the weights and the biases | calculate the score of each example (`forward`) |
| the gradients for the weights and the biases | work out the gradients (`backward`) |
| the inputs it was last given | change the weights by gradient descent (`update`) |

---
# Part 2 · The data

▶ Run the cell below to prepare the data. It is the simulated data from LELA60331 Weeks 5 and 7: 50 texts, each with two features (the log counts of positive and of negative words) and a label `y` (1 = positive, 0 = negative).

As in LELA60331 Week 7, Part 4, the features are stored as a **matrix** `X` with **one row per text** and one column per feature, so that `X[k]` is the list `[x1, x2]` for text `k`.

In [ ]:
# ▶ Run this cell: it makes the data matrix X and the labels y
import math
import random
import matplotlib.pyplot as plt

random.seed(10)
x = [[], []]                           # x[0]: feature 1 of every text; x[1]: feature 2
y = []
for i in range(50):
    if random.random() > 0.5:          # a negative text: fewer positive than negative words
        x[0].append(random.gauss(2, 1))
        x[1].append(random.gauss(3, 1))
        y.append(0)
    else:                              # a positive text: more positive than negative words
        x[0].append(random.gauss(3, 1))
        x[1].append(random.gauss(2, 1))
        y.append(1)

X = [[x[0][k], x[1][k]] for k in range(len(y))]          # one row [x1, x2] per text

print(len(X), "rows,", len(X[0]), "columns;", sum(y), "positive texts")
print("first text:", X[0], "label:", y[0])

---
# Part 3 · A linear layer, one method at a time

A `Linear` layer does the matrix-vector multiplication from LELA60331 Week 7. It has `n_out` **outputs**, and for each example it gives one score per output. Logistic regression needs just one output: the score that the text is positive. (Other models need more, and that is why the layer allows several.)

We build it in four steps. At each step you work out a small example first, then add one method to the class:

| Step | Method | What it does |
|---|---|---|
| 1 | `__init__` | makes the weights and biases *(done for you)* |
| 2 | `forward(X)` | the scores of a list of examples |
| 3 | `backward(grad)` | the gradients for the weights and biases |
| 4 | `update(lr)` | changes the weights and biases by gradient descent |

## Step 1 · Making a layer

Here is the class with just its constructor. A layer with `n_in` inputs and `n_out` outputs has:

* `self.weight`: a matrix with **one row per output** and one column per input feature
* `self.bias`: a list with one bias per output

Both start at random numbers between 0 and 1, as in LELA60331 Week 7. Run the cell.

In [ ]:
class Linear:
    """A layer that gives n_out scores for each example: one row of weights and one bias per output."""
    def __init__(self, n_in, n_out):
        self.weight = [[random.random() for j in range(n_in)] for c in range(n_out)]
        self.bias = [random.random() for c in range(n_out)]

layer = Linear(2, 3)
print(layer.weight)
print(layer.bias)

## Step 2 · The forward pass

For one example `x`, the score of output `c` is the **dot product** of row `c` of the weights with `x`, plus bias `c`.

### The tools: `dot` and `mat_vec`

We need two helper functions that you wrote before: `dot` (Week 4), and `mat_vec` (LELA60331 Week 7, Activity 10), which takes the dot product of **each row** of a matrix with a vector, so it gives all the outputs' dot products at once.

🔮 **Predict**: what will `dot([1, 2], [3, 4])` give? And `mat_vec([[1, 0], [0, 1], [1, 1]], [2, 3])`? Run the cell to define them and check.

In [ ]:
# ▶ Run this cell: the helper functions for the forward pass
def dot(a, b):
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def mat_vec(matrix, vector):
    # the dot product of each row of the matrix with the vector
    return [dot(row, vector) for row in matrix]

print(dot([1, 2], [3, 4]))
print(mat_vec([[1, 0], [0, 1], [1, 1]], [2, 3]))

**💡 Answer.** `dot([1, 2], [3, 4])` is 1 × 3 + 2 × 4 = **11**. `mat_vec([[1, 0], [0, 1], [1, 1]], [2, 3])` takes the dot product of each row with `[2, 3]`: **`[2, 3, 5]`**.

### A worked example

Here are weights and biases chosen by hand so that the numbers are easy. `mat_vec` gives the dot products (the middle column); then we add each output's bias:

| output `c` | weights `weight[c]` | dot product with `x = [2, 3]` | bias `bias[c]` | score |
|---|---|---|---|---|
| 0 | `[1, 0]` | 1 × 2 + 0 × 3 = 2 | 0 | **2** |
| 1 | `[0, 1]` | 0 × 2 + 1 × 3 = 3 | 10 | **13** |
| 2 | `[1, 1]` | 1 × 2 + 1 × 3 = 5 | 100 | **105** |

### 🏋️ Exercise 1 · Practice — The scores of one example, by hand

Using the weights and biases from the table:

1. `sums`: `mat_vec` with `W` and `x_example` (one number per output)
2. `z`: complete the loop so that it adds each output's bias

In [ ]:
W = [[1, 0], [0, 1], [1, 1]]
b = [0, 10, 100]
x_example = [2, 3]

sums = mat_vec(W, x_example)                   # 1.
z = []
for c in range(len(sums)):
    z.append(sums[c] + b[c])                   # 2.
print(sums, z)                                 # you should get [2, 3, 5] [2, 13, 105]

In [ ]:
# ✅ Check
check("1. sums", lambda: sums == [2, 3, 5])
check("2. z", lambda: z == [2, 13, 105])

### 🏋️ Exercise 2 · Practice — Adding `forward` to the class

Now put the same steps in a method. `forward(X)` takes a **list of examples** (one row each, like `X`) and returns a matrix with **one row per example** and one score per output. The loop over `X` does, for each example in turn, exactly what you did for one example in Exercise 1. It also keeps `X` in `self.inputs`, because the backward pass will need it.

The cell repeats the class from Step 1, with the new method added. Fill in the blanks 1–2, the same calculations as in Exercise 1 but with `self.weight` and `self.bias`, then run the cell to replace the old class with the new one.

The cell tests it first on the one example from Exercise 1, then on two examples.

In [ ]:
class Linear:
    """A layer that gives n_out scores for each example: one row of weights and one bias per output."""
    def __init__(self, n_in, n_out):
        self.weight = [[random.random() for j in range(n_in)] for c in range(n_out)]
        self.bias = [random.random() for c in range(n_out)]

    def forward(self, X):
        self.inputs = X                                    # kept for the backward pass
        scores = []
        for x in X:                                        # each example
            z = mat_vec(self.weight, x)                    # 1. one number per output
            for c in range(len(z)):
                z[c] += self.bias[c]                       # 2. plus that output's bias
            scores.append(z)
        return scores


layer = Linear(2, 3)
layer.weight = [[1, 0], [0, 1], [1, 1]]         # set the weights and biases by hand, to test it
layer.bias = [0, 10, 100]
print(layer.forward([[2, 3]]))                   # one example: you should get [[2, 13, 105]], as in Exercise 1
print(layer.forward([[2, 3], [1, 1]]))           # two examples: you should get [[2, 13, 105], [1, 11, 102]]

In [ ]:
# ✅ Check
def _layer():
    l = Linear(2, 3)
    l.weight = [[1, 0], [0, 1], [1, 1]]
    l.bias = [0, 10, 100]
    return l
check("1–2. forward, one example", lambda: _layer().forward([[2, 3]]) == [[2, 13, 105]])
check("1–2. forward, two examples", lambda: _layer().forward([[2, 3], [1, 1]]) == [[2, 13, 105], [1, 11, 102]])
check("forward keeps the inputs", lambda: (lambda l: (l.forward([[2, 3]]), l.inputs)[1])(_layer()) == [[2, 3]])

## Step 3 · The backward pass

The backward pass works out the gradients for the weights and biases. It is given `grad`, which has the same shape as the scores: `grad[k][c]` says how much the loss changes when the score of output `c` for example `k` goes up. (In the after-class workbook you will see where `grad` comes from: it is the error from LELA60331 Week 5.)

The rule is the one from LELA60331 Week 7:

* the gradient for `weight[c][j]` is `grad[k][c] * inputs[k][j]`: the error for that output, times the feature
* the gradient for `bias[c]` is `grad[k][c]`

### A worked example: one example

Here is the calculation for **one** example, `x = [2, 3]`, whose gradients for the three outputs are `[1, 0, 2]`:

| | feature 0 (`x[0]` = 2) | feature 1 (`x[1]` = 3) | bias |
|---|---|---|---|
| output 0 (grad 1) | 1 × 2 = **2** | 1 × 3 = **3** | **1** |
| output 1 (grad 0) | 0 × 2 = **0** | 0 × 3 = **0** | **0** |
| output 2 (grad 2) | 2 × 2 = **4** | 2 × 3 = **6** | **2** |

### 🏋️ Exercise 3 · Practice — The gradients for one example, by hand

Make the gradient matrix in the table with nested loops: an outer loop for the rows (one per output) and an inner loop for the columns (one per feature), as for matrices in Week 4:

1. complete the inner loop, so that row `c` holds `g[c] * x_example[j]` for each feature `j`
2. `bias_grad_example`: the gradients for the biases. *Hint:* they are just the gradients `g`; `list(g)` makes a copy.

In [ ]:
x_example = [2, 3]
g = [1, 0, 2]                     # the gradient of each of the three outputs, for this example

weight_grad_example = []
for c in range(len(g)):           # each output: one row
    row = []
    for j in range(len(x_example)):              # each feature: one column
        row.append(g[c] * x_example[j])          # 1.
    weight_grad_example.append(row)
bias_grad_example = list(g)                      # 2.
print(weight_grad_example, bias_grad_example)    # you should get [[2, 3], [0, 0], [4, 6]] [1, 0, 2]

In [ ]:
# ✅ Check
check("1. weight_grad_example", lambda: weight_grad_example == [[2, 3], [0, 0], [4, 6]])
check("2. bias_grad_example", lambda: bias_grad_example == [1, 0, 2])

### A worked example: two examples

With several examples, each example's contribution is worked out as above, and the contributions are **added up**. Take two examples, `[2, 3]` with gradients `[1, 0, 0]`, and `[1, 1]` with gradients `[0, 0, 2]`:

| | example 0's contribution | example 1's contribution | total weight gradient | total bias gradient |
|---|---|---|---|---|
| output 0 | 1 × `[2, 3]` = `[2, 3]` | 0 × `[1, 1]` = `[0, 0]` | **`[2, 3]`** | 1 + 0 = **1** |
| output 1 | 0 × `[2, 3]` = `[0, 0]` | 0 × `[1, 1]` = `[0, 0]` | **`[0, 0]`** | 0 + 0 = **0** |
| output 2 | 0 × `[2, 3]` = `[0, 0]` | 2 × `[1, 1]` = `[2, 2]` | **`[2, 2]`** | 0 + 2 = **2** |

So in the method, the gradients start as 0s, and each example's contribution is **added** to them with `+=`.

### One more tool: `transpose`

The method's last line (done for you) returns the gradients for the layer's **inputs**, which are needed when one layer feeds into another. It uses the **transpose** of the weights: the same numbers with the rows turned into columns (LELA60331 Week 7, Activities 6, 7 and 12).

🔮 **Predict**: what is the transpose of `[[1, 2, 3], [4, 5, 6]]`? Run the cell to define `transpose` and check.

In [ ]:
# ▶ Run this cell: the helper function for the backward pass
def transpose(matrix):
    # the rows of the matrix turned into columns
    return [[row[j] for row in matrix] for j in range(len(matrix[0]))]

print(transpose([[1, 2, 3], [4, 5, 6]]))

**💡 Answer.** **`[[1, 4], [2, 5], [3, 6]]`**: the 2 rows of 3 become 3 rows of 2. Row `j` of the transpose is column `j` of the original.

### 🏋️ Exercise 4 · Practice — Adding `backward` to the class

There is one loop for the examples, one for the outputs and one for the features. Fill in the blanks 1–2: the same calculations as in Exercise 3, with `grad[k][c]` and `self.inputs[k][j]` for example `k`.

The cell tests it first on the one example from Exercise 3, then on the two examples from the table above.

In [ ]:
class Linear:
    """A layer that gives n_out scores for each example: one row of weights and one bias per output."""
    def __init__(self, n_in, n_out):
        self.weight = [[random.random() for j in range(n_in)] for c in range(n_out)]
        self.bias = [random.random() for c in range(n_out)]

    def forward(self, X):
        self.inputs = X
        scores = []
        for x in X:
            z = mat_vec(self.weight, x)
            for c in range(len(z)):
                z[c] += self.bias[c]
            scores.append(z)
        return scores

    def backward(self, grad):
        n_out = len(self.weight)
        n_in = len(self.weight[0])
        self.weight_grad = [[0] * n_in for c in range(n_out)]     # start from 0s
        self.bias_grad = [0] * n_out
        for k in range(len(self.inputs)):                  # each example
            for c in range(n_out):                         # each output
                for j in range(n_in):                      # each input feature
                    self.weight_grad[c][j] += grad[k][c] * self.inputs[k][j]       # 1.
                self.bias_grad[c] += grad[k][c]                                    # 2.
        # the gradients for the inputs, for a layer that comes before this one (done for you)
        return [mat_vec(transpose(self.weight), gr) for gr in grad]


layer = Linear(2, 3)
layer.weight = [[1, 0], [0, 1], [1, 1]]
layer.bias = [0, 10, 100]
layer.forward([[2, 3]])
layer.backward([[1, 0, 2]])
print(layer.weight_grad, layer.bias_grad)        # the same as Exercise 3: [[2, 3], [0, 0], [4, 6]] [1, 0, 2]
layer.forward([[2, 3], [1, 1]])
layer.backward([[1, 0, 0], [0, 0, 2]])
print(layer.weight_grad, layer.bias_grad)        # the table above: [[2, 3], [0, 0], [2, 2]] [1, 0, 2]

In [ ]:
# ✅ Check
def _layer():
    l = Linear(2, 3)
    l.weight = [[1, 0], [0, 1], [1, 1]]
    l.bias = [0, 10, 100]
    return l
def _one():
    l = _layer(); l.forward([[2, 3]]); l.backward([[1, 0, 2]])
    return l.weight_grad == [[2, 3], [0, 0], [4, 6]] and l.bias_grad == [1, 0, 2]
check("1–2. backward, one example: the same as Exercise 3", _one)
def _two():
    l = _layer(); l.forward([[2, 3], [1, 1]]); l.backward([[1, 0, 0], [0, 0, 2]])
    return l.weight_grad == [[2, 3], [0, 0], [2, 2]] and l.bias_grad == [1, 0, 2]
check("1–2. backward, two examples: their contributions are added", _two)
def _twice():
    l = _layer(); l.forward([[2, 3]]); l.backward([[1, 0, 2]]); l.backward([[1, 0, 2]])
    return l.weight_grad == [[2, 3], [0, 0], [4, 6]]
check("backward starts from 0s each time", _twice)

## Step 4 · The update

Finally, gradient descent, exactly as in LELA60331 Weeks 4, 5 and 7: **new value = old value − learning rate × gradient**, for every weight and every bias.

For example, with a learning rate of 0.5 and the gradients from the two-example table in Step 3:

| | old value | gradient | new value |
|---|---|---|---|
| `weight[0][1]` | 0 | 3 | 0 − 0.5 × 3 = **−1.5** |
| `weight[2][0]` | 1 | 2 | 1 − 0.5 × 2 = **0.0** |
| `bias[2]` | 100 | 2 | 100 − 0.5 × 2 = **99.0** |

### 🏋️ Exercise 5 · Practice — Adding `update` to the class

Fill in the blanks 1–2, one for the weights and one for the biases.

In [ ]:
class Linear:
    """A layer that gives n_out scores for each example: one row of weights and one bias per output."""
    def __init__(self, n_in, n_out):
        self.weight = [[random.random() for j in range(n_in)] for c in range(n_out)]
        self.bias = [random.random() for c in range(n_out)]

    def forward(self, X):
        self.inputs = X
        scores = []
        for x in X:
            z = mat_vec(self.weight, x)
            for c in range(len(z)):
                z[c] += self.bias[c]
            scores.append(z)
        return scores

    def backward(self, grad):
        n_out = len(self.weight)
        n_in = len(self.weight[0])
        self.weight_grad = [[0] * n_in for c in range(n_out)]
        self.bias_grad = [0] * n_out
        for k in range(len(self.inputs)):
            for c in range(n_out):
                for j in range(n_in):
                    self.weight_grad[c][j] += grad[k][c] * self.inputs[k][j]
                self.bias_grad[c] += grad[k][c]
        return [mat_vec(transpose(self.weight), gr) for gr in grad]

    def update(self, lr):
        for c in range(len(self.weight)):
            for j in range(len(self.weight[c])):
                self.weight[c][j] -= lr * self.weight_grad[c][j]            # 1.
            self.bias[c] -= lr * self.bias_grad[c]                          # 2.


layer = Linear(2, 3)
layer.weight = [[1, 0], [0, 1], [1, 1]]
layer.bias = [0, 10, 100]
layer.forward([[2, 3], [1, 1]])
layer.backward([[1, 0, 0], [0, 0, 2]])
layer.update(0.5)
print(layer.weight, layer.bias)        # you should get [[0.0, -1.5], [0.0, 1.0], [0.0, 0.0]] [-0.5, 10.0, 99.0]

In [ ]:
# ✅ Check
def _update():
    l = Linear(2, 3)
    l.weight = [[1, 0], [0, 1], [1, 1]]
    l.bias = [0, 10, 100]
    l.forward([[2, 3], [1, 1]])
    l.backward([[1, 0, 0], [0, 0, 2]])
    l.update(0.5)
    return l.weight == [[0.0, -1.5], [0.0, 1.0], [0.0, 0.0]] and l.bias == [-0.5, 10.0, 99.0]
check("1–2. update", _update)

That's the whole layer: four methods, each of which you have tested on its own.

---
## ✏️ Summary

You can now:

* explain why a program with many shared global names is fragile, and how a class helps
* build a class one method at a time, testing each method on a small example
* write a `Linear` layer with a **forward** pass, a **backward** pass and an **update**

**After class:** work through `LELA60341_Wk8_after_class`. There you write the loss, and use it to train a logistic regression model built from the layer you have made here.